In [ ]:
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
E_J = 90 *2pi /2  # ħ * GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
κ=1/20; # 1/ns from Jeffrey et al.

ω_nl=sqrt(4*C_inv*E_J)

In [ ]:
n=2000
t_max = 3/κ
bath=Exp_decay
bath_params = (ω_0=50*2*pi, ω_cut_off=20*2*pi)

In [ ]:
ϕ,q,t_list = One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=E_J, κ=κ, C_inv=C_inv, ω=ω_nl, A=0, β=∞, energy=false,fast_no_chain=true)

In [ ]:
F,freqs=FFT_perso(ϕ, t_list)
f_shifted=abs(freqs[argmax(abs.(F))])
plot(freqs,abs.(F))
vline!([f_shifted], linestyle=:dash, color=:red, label="f_shifted")
plot!(xlims=(4,8))

In [ ]:
f_shifted*2*pi

In [ ]:
Lamb_shift=f_shifted-ω_nl/2pi

In [ ]:
ϕ,q,t_list = One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=E_J, κ=κ, C_inv=C_inv, ω=ω_nl-Lamb_shift*2*pi, A=1, β=∞, energy=false,fast_no_chain=true)

In [ ]:
plot(t_list*κ,ϕ)

In [ ]:
ϕ,q,t_list = One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=E_J, κ=κ, C_inv=C_inv, ω=ω_nl+Lamb_shift*2*pi, A=1, β=∞, energy=false,fast_no_chain=true)

In [ ]:
plot(t_list*κ,ϕ)

In [ ]:
ϕ_chain,ϕ_dot_chain,t_list,Mq_chain, Mϕ_chain=One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=E_J, κ=κ, C_inv=C_inv, ω=ω_nl+Lamb_shift*2*pi, A=1, β=∞, energy=false,emission=true)

In [ ]:
A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain)


# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ_chain
S_dot = P_chain \ ϕ_dot_chain

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain)
vline!([ω_nl/2pi],linestyle=:dash, color=:red)
annotate!(ω_nl/2pi-.2,1e-0, Plots.text("mode "*L"a", 8, :red, rotation = 90 ))
plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 10.2))

In [ ]:
using ProgressBars

In [ ]:
ω_drive=.9ω_nl:.05:1.1ω_nl
Energies_normal_modes_chain_list=Array{Float64}(undef, n, length(ω_drive))

for i in ProgressBar(1:length(ω_drive))
    ω_d=ω_drive[i]
    ϕ_chain,ϕ_dot_chain,t_list,Mq_chain, Mϕ_chain=One_d_non_linear_driven(n, bath, bath_params, t_max;E_J=E_J, κ=κ, C_inv=C_inv, ω=ω_d, A=1, β=∞, energy=false,emission=true)

    # Solve P_chain * S = ϕ for S instead of inverting
    S = P_chain \ ϕ_chain
    S_dot = P_chain \ ϕ_dot_chain

    # Energy in normal modes
    Energies_normal_modes_chain_list[:,i] = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;
end

In [ ]:
heatmap(ωk_chain/2pi,(ω_drive.-ω_nl)/2pi,log10.(Energies_normal_modes_chain_list'))
hline!([Lamb_shift], linestyle=:dash, color=:red)
plot!(ωk_chain/2pi, (ωk_chain.-ω_nl)/2pi, linestyle=:dash, lw=1, color=:black)
plot!(ylabel="Drive frequency "*L"(\omega_d-\omega_{0})/2\pi"*" (GHz)", xlabel="Frequency (GHz)",
 legend=false,
 xlims=(0.1, 10.2), ylims=(minimum(ω_drive.-ω_nl)/2pi, maximum(ω_drive.-ω_nl)/2pi),
 colorbar=true,
 colorbar_title="\n"*L"\log_{10}"*"(Energy)",
 topmargin=10mm, right_margin=15mm,)
